In [1]:
import numpy as np

import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
from astropy.convolution import convolve, Gaussian2DKernel
import  astropy.convolution as convolution
from astropy.wcs import WCS
from reproject import reproject_interp

import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

from astropy.coordinates import SkyCoord
import astropy.units as u

plt.rcParams['font.size'] = 35
plt.rcParams['axes.labelsize'] = 35
plt.rcParams['xtick.labelsize'] = 35
plt.rcParams['ytick.labelsize'] = 35
plt.rcParams['legend.fontsize'] = 35
plt.rcParams['figure.titlesize'] = 35
plt.rcParams['axes.labelweight'] = 'heavy'
plt.rcParams['axes.linewidth'] = 3
plt.rcParams['xtick.major.size'] = 20
plt.rcParams['xtick.minor.size'] = 8
plt.rcParams['ytick.major.size'] = 20
plt.rcParams['ytick.minor.size'] = 8
plt.rcParams["contour.linewidth"] =5



In [2]:
RA=247.0900625 #edisk
Dec=-24.6067583#edisk

In [3]:
def load_fits(filename):
    with fits.open(filename) as hdul:
        return hdul[0].data, hdul[0].header

def save_fits(data, header, filename):
    hdu = fits.PrimaryHDU(data, header=header)
    hdu.writeto(filename, overwrite=True)

def get_pixel_scale(header):
    # Assume pixel scale is in arcsec/pixel, read from FITS header
    if 'CD1_1' in header:
        pixel_scale = abs(header['CD1_1']) * 3600  # Convert degrees to arcsec
    elif 'CDELT1' in header:
        pixel_scale = abs(header['CDELT1']) * 3600  # Convert degrees to arcsec
    else:
        raise ValueError("Pixel scale information not found in FITS header.")
    return pixel_scale



def degrade_resolution(image, current_fwhm_arcsec, target_fwhm_arcsec, pixel_scale):
  
    # Compute the Gaussian kernel sigma for convolution
    if target_fwhm_arcsec > current_fwhm_arcsec:
        sigma = (np.sqrt(target_fwhm_arcsec**2 - current_fwhm_arcsec**2)/2.35482) / pixel_scale
        kernel = Gaussian2DKernel(sigma)
        degraded_image = convolution.convolve_fft(image, kernel, preserve_nan=True, normalize_kernel=True)


    else:
        print("Target resolution must be lower than the current resolution.")
        degraded_image = image
    
    return degraded_image

In [4]:
image, header = load_fits('H200S11_4.18_linemap.fits')
pixel_scale = get_pixel_scale(header)
current_fwhm_arcsec = 0.2 #(0.033*6.91) + 0.106 # Set the current resolution FWHM in arcsec
target_fwhm_arcsec = (0.033*17.033) + 0.106  # Set the target lower resolution FWHM in arcsec
degraded_image = degrade_resolution(image, current_fwhm_arcsec, target_fwhm_arcsec, pixel_scale)

save_fits(degraded_image, header, 'H2_S11_degraded_image.fits')

FileNotFoundError: [Errno 2] No such file or directory: 'H200S11_4.18_linemap.fits'

In [ ]:
image, header = load_fits('cont_41.fits')
pixel_scale = get_pixel_scale(header)
current_fwhm_arcsec = 0.2 #(0.033*6.91) + 0.106 # Set the current resolution FWHM in arcsec
target_fwhm_arcsec = (0.033*17.033) + 0.106  # Set the target lower resolution FWHM in arcsec
degraded_image = degrade_resolution(image, current_fwhm_arcsec, target_fwhm_arcsec, pixel_scale)

save_fits(degraded_image, header, 'Cont_degraded_image.fits')

In [ ]:
target_fwhm_arcsec

In [ ]:
image, header = load_fits('H200S5_6.9095_linemap.fits')
pixel_scale = get_pixel_scale(header)
current_fwhm_arcsec = (0.033*6.91) + 0.106 # Set the current resolution FWHM in arcsec
target_fwhm_arcsec = (0.033*17.033) + 0.106  # Set the target lower resolution FWHM in arcsec
degraded_image = degrade_resolution(image, current_fwhm_arcsec, target_fwhm_arcsec, pixel_scale)

save_fits(degraded_image, header, 'H2_S5_degraded_image.fits')

In [ ]:
plt.rcParams["contour.linewidth"] =5
plt.rcParams['axes.facecolor'] = 'black'
hdu2 = fits.open('H200S1_linemap.fits')[0]
hdu1 = fits.open('H2_S11_degraded_image.fits')[0]

# Extract the data arrays and WCS objects from the HDUs
data1 = hdu1.data
wcs1 = WCS(hdu1.header)
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu1.header['CDELT1']*3600

sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs1.world_to_pixel(sky) 


v1=5e-7
v2=3e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu1.header)
ax = fig.add_subplot(111, projection=proj)

data2=np.array(data2)
data2=np.nan_to_num(data2)

im = ax.imshow(np.log10(data1), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)
ax.contour(data2,transform=ax.get_transform(wcs2),levels=np.max(data2)*np.array([0.4,0.6,0.8,0.99]),colors='lime')


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")

ax.text(60,88,'H$_2$ 0-0 S(11)',color='khaki',size=30,weight='heavy')
ax.text(60,82,'H$_2$ 0-0 S(1)',color='lime',size=30,weight='heavy')

ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
# ra_offset.coord_wrap = 180 # avoid wrapping
plt.savefig('Figure6_all_native.png')

In [ ]:
np.max(data2)

In [ ]:

hdu2 = fits.open('H200S1_linemap.fits')[0]
hdu1 = fits.open('H2_S11_degraded_image.fits')[0]


from reproject import reproject_interp
data2, footprint = reproject_interp(hdu2, hdu1.header)

# Extract the data arrays and WCS objects from the HDUs
data1 = hdu1.data
wcs1 = WCS(hdu1.header)
#data2 = hdu2.data
#wcs2 = WCS(hdu2.header)
ps=hdu1.header['CDELT1']*3600

sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs1.world_to_pixel(sky) 


v1=5e-7
v2=3e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu1.header)
ax = fig.add_subplot(111, projection=proj)

data2=np.array(data2)
data2=np.nan_to_num(data2)

im = ax.imshow(np.log10(data1), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)
ax.contour(data2,levels=np.max(data2)*np.array([0.4,0.6,0.8,0.99]),colors='lime')


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ax.text(60,88,'H$_2$ 0-0 S(11)',color='khaki',size=30,weight='heavy')
ax.text(60,82,'H$_2$ 0-0 S(1)',color='lime',size=30,weight='heavy')

ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
# ra_offset.coord_wrap = 180 # avoid wrapping
plt.savefig('Figure6_S1_reprojected_to_S11.png')

In [ ]:
np.max(data2)

In [ ]:
hdu2 = fits.open('H200S1_linemap.fits')[0]
hdu1 = fits.open('H2_S11_degraded_image.fits')[0]

data1 = hdu1.data
wcs2 = WCS(hdu2.header)

data2, footprint = reproject_interp(hdu2, hdu1.header)

fig = plt.figure(figsize=[12, 12])
proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)

ps=hdu2.header['CDELT1']*3600

ax.contour(data2,origin='lower',levels=np.max(data2)*np.array([0.4,0.6,0.8,0.99]),colors='lime')
ax.imshow(hdu2.data,transform=ax.get_transform(wcs2))

# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))

print(ps)

In [ ]:

hdu2 = fits.open('H200S1_linemap.fits')[0]
hdu1 = fits.open('H2_S11_degraded_image.fits')[0]


from reproject import reproject_interp
data1, footprint = reproject_interp(hdu1, hdu2.header)

# Extract the data arrays and WCS objects from the HDUs
#data1 = hdu1.data
#wcs1 = WCS(hdu1.header)
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600

sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=5e-7
v2=3e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)

data2=np.array(data2)
data2=np.nan_to_num(data2)

im = ax.imshow(np.log10(data1), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)
ax.contour(data2,levels=np.max(data2)*np.array([0.4,0.6,0.8,0.99]),colors='lime')



# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]


ax.text(34,48,'H$_2$ 0-0 S(11)',color='khaki',size=30,weight='heavy')
ax.text(34,44,'H$_2$ 0-0 S(1)',color='lime',size=30,weight='heavy')


dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
# ra_offset.coord_wrap = 180 # avoid wrapping
plt.savefig('Figure6_S11_reprojected_to_S1.png')

In [ ]:
np.max(data2)

In [ ]:
hdu2 = fits.open('H200S1_linemap.fits')[0]
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600
sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=1e-6
v2=1e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)



im = ax.imshow(np.log10(data2), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))

In [ ]:
hdu2 = fits.open('H200S11_linemap.fits')[0]
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600
sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=5e-7
v2=3e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)



im = ax.imshow(np.log10(data2), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
plt.tight_layout()
plt.savefig('H2_S11.png')

In [ ]:
hdu2 = fits.open('H2_S11_degraded_image.fits')[0]
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600
sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=5e-7
v2=3e-5

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)



im = ax.imshow(np.log10(data2), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)



# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
plt.tight_layout()
plt.savefig('H2_S11.png')

In [ ]:
hdu2 = fits.open('cont_41.fits')[0]
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600
sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=0.01
v2=1

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)



im = ax.imshow(np.log10(data2), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
plt.tight_layout()
plt.savefig('H2_S11.png')

In [ ]:
hdu2 = fits.open('Cont_degraded_image.fits')[0]
data2 = hdu2.data
wcs2 = WCS(hdu2.header)
ps=hdu2.header['CDELT1']*3600
sky=SkyCoord(RA, Dec, unit="deg")

xa, ya = wcs2.world_to_pixel(sky) 


v1=0.01
v2=1

region_center = SkyCoord(RA, Dec, unit=u.deg, frame='icrs' )

fig = plt.figure(figsize=[12, 12])

proj = WCS(hdu2.header)
ax = fig.add_subplot(111, projection=proj)



im = ax.imshow(np.log10(data2), origin='lower',cmap='magma',alpha=1, vmin=np.log10(v1),vmax=np.log10(v2))
plt.colorbar(im)


# Remove the absolute coordinates
ra = ax.coords["ra"]
dec = ax.coords["dec"]
ra.set_ticks_visible(False)
ra.set_ticklabel_visible(False)
dec.set_ticks_visible(False)
dec.set_ticklabel_visible(False)
ra.set_axislabel("")
dec.set_axislabel("")

# Create an overlay with relative coordinates
aframe = region_center.skyoffset_frame()
overlay = ax.get_coords_overlay(aframe)
ra_offset = overlay["lon"]
dec_offset = overlay["lat"]



dec_offset.set_axislabel(f"Offset Dec (arcsec)")
ra_offset.set_axislabel("Offset RA (arcsec) ")


ra_offset.set_major_formatter("s")
dec_offset.set_major_formatter("s")
ra_offset.set_ticks_position("b")
ra_offset.set_ticklabel_position("b")
dec_offset.set_ticks_position("l")
dec_offset.set_ticklabel_position("l")
ra_offset.set_axislabel_position("b")
dec_offset.set_axislabel_position("l")
#ra_offset.coord_wrap = 180
plt.ylim(ya-(5.1/np.abs(ps)),ya+(5.1/np.abs(ps)))
plt.xlim(xa-(5.1/np.abs(ps)),xa+(5.1/np.abs(ps)))
plt.tight_layout()
plt.savefig('H2_S11.png')